# 4. scVI-style cVAE Batch Correction

Батч-коррекция COMPASS PreTrainer эмбеддингов с помощью **Conditional VAE**.

**Почему cVAE а не scVI?**
scVI предполагает count data (Negative Binomial), наши эмбеддинги —
непрерывные 4,224-dim float32. Мы используем ту же архитектуру VAE,
но с **Gaussian likelihood** (MSE reconstruction).

**Pipeline:**
1. Load cohort from `data/interim_PT/`
2. Generate dummy split (`Split_dummy`: 60% train / 20% test / 20% NaN)
3. Fit cVAE on **train** embeddings only
4. Transform **both** train and test → 128-dim batch-invariant latent
5. Store in `.obsm["FM_COMPASS_PT_embedding_scVI_corrected"]`
6. Save to `data/processed/`

**Architecture:**
```
Encoder (batch-FREE):  4224 → 512 → 256 → (μ₁₂₈, logσ²₁₂₈)
Decoder (batch-COND):  (128 + n_batches) → 256 → 512 → 4224
Loss:  MSE_recon + β · KL(q||p)
```

## 0. Environment Setup

In [ ]:
# !rm -rf batchcor-rna-embeds/  # if already cloned
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY, SCVI_LATENT_DIM,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import CVAECorrector, CVAEConfig

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}, PyTorch: {}", DEVICE, torch.__version__)

## 1. Mount Drive & Define Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Source: PreTrainer embeddings
DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
# Destination: batch-corrected results
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_COL = "Split_dummy"

logger.info("Source (interim_PT): {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])
logger.info("Destination (processed): {}", DATA_PROCESSED_DIR)

## 2. Process All Cohorts

For each cohort:
1. Load from `interim_PT`
2. Add `Split_dummy` (60/20/20)
3. Fit cVAE on train → transform train + test
4. Store corrected embeddings
5. Save to `processed/`

In [ ]:
cvae_config = CVAEConfig(
    latent_dim=SCVI_LATENT_DIM,
    hidden_dims=(512, 256),
    n_epochs=150,
    batch_size=128,
    lr=1e-3,
    beta_max=1.0,
    warmup_fraction=0.3,
    dropout=0.2,
    seed=SEED,
)

all_histories = {}

for cohort_name in COHORT_CANCER_CODE:
    src_path  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    dest_path = DATA_PROCESSED_DIR  / f"{cohort_name}.adata.zarr"

    if not src_path.exists():
        logger.warning("Cohort not found in interim_PT, skipping: {}", cohort_name)
        continue

    if dest_path.exists():
        logger.info("Already in processed, skipping: {}", cohort_name)
        continue

    logger.info("\n" + "=" * 70)
    logger.info("Processing: {}", cohort_name)
    logger.info("=" * 70)

    # Load
    adata = load_cohort(src_path)
    logger.info("Loaded: {} samples x {} genes", adata.n_obs, adata.n_vars)

    # Check embedding exists
    if COMPASS_PT_EMBEDDING_KEY not in adata.obsm:
        logger.error("Missing '{}' in obsm, skipping", COMPASS_PT_EMBEDDING_KEY)
        del adata
        continue

    # Add dummy split
    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)

    logger.info("Batch distribution:\n{}", adata.obs[BATCH_COL].value_counts())
    logger.info("Split: train={}, test={}, NaN={}",
        train_mask.sum(), test_mask.sum(),
        adata.n_obs - train_mask.sum() - test_mask.sum(),
    )

    # Extract embeddings
    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch_labels_all = adata.obs[BATCH_COL].astype(str).values

    # Fit on TRAIN only
    corrector = CVAECorrector(config=cvae_config)
    corrector.fit(
        X=emb_all[train_mask],
        batch_labels=batch_labels_all[train_mask],
    )
    all_histories[cohort_name] = corrector.history_

    # Transform ALL annotated samples (train + test)
    # NaN samples also get transformed for completeness
    corrected = corrector.transform(emb_all)

    # Store in obsm
    adata.obsm[SCVI_CORRECTED_KEY] = corrected

    # Verify
    logger.info("Corrected embedding shape: {}, dtype: {}",
        corrected.shape, corrected.dtype,
    )
    assert corrected.shape == (adata.n_obs, SCVI_LATENT_DIM)
    assert np.isfinite(corrected).all(), "NaN/Inf detected in corrected embeddings!"

    # Save to processed
    save_adata_zarr(adata, dest_path)
    logger.info("Saved to: {}", dest_path)

    del adata, corrected
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed.")
logger.info("Processed contents: {}", [p.name for p in sorted(DATA_PROCESSED_DIR.glob('*.adata.zarr'))])

## 3. Training Loss Curves

In [ ]:
n_cohorts = len(all_histories)
if n_cohorts == 0:
    logger.warning("No histories to plot (all cohorts were skipped).")
else:
    fig, axes = plt.subplots(n_cohorts, 3, figsize=(18, 4 * n_cohorts))
    if n_cohorts == 1:
        axes = axes[np.newaxis, :]

    for row, (name, hist) in enumerate(all_histories.items()):
        epochs = range(1, len(hist.total) + 1)

        axes[row, 0].plot(epochs, hist.recon, color='#2196F3', linewidth=1.5)
        axes[row, 0].set_title(f"{name} — Reconstruction (MSE)", fontsize=10)
        axes[row, 0].set_xlabel("Epoch")

        axes[row, 1].plot(epochs, hist.kl, color='#FF5722', linewidth=1.5)
        axes[row, 1].set_title(f"{name} — KL Divergence", fontsize=10)
        axes[row, 1].set_xlabel("Epoch")

        axes[row, 2].plot(epochs, hist.beta_schedule, color='#4CAF50', linewidth=1.5)
        axes[row, 2].set_title(f"{name} — β Schedule", fontsize=10)
        axes[row, 2].set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

## 4. Before / After UMAP Comparison

Сравнение UMAP до и после коррекции для первой доступной когорты.

In [ ]:
from matplotlib.lines import Line2D
from umap import UMAP


def plot_umap_panel(coords, labels, title, ax, palette=None):
    """Scatter UMAP colored by labels."""
    unique = sorted(labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique))
    cmap = dict(zip(unique, palette))
    colors = [cmap[l] for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6,
               edgecolors="none", rasterized=True)
    ax.set_title(title, fontsize=10, fontweight="bold")
    if len(unique) <= 12:
        handles = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cmap[l], markersize=7, label=str(l))
                   for l in unique]
        ax.legend(handles=handles, fontsize=6, loc="best",
                  framealpha=0.8, edgecolor="#ccc")


# Pick first available cohort
test_cohort = None
for name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_DIR / f"{name}.adata.zarr"
    if p.exists():
        test_cohort = name
        break

if test_cohort:
    adata = load_cohort(DATA_PROCESSED_DIR / f"{test_cohort}.adata.zarr")

    # UMAP on raw PT embeddings
    emb_raw = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    umap_raw = UMAP(n_components=2, random_state=SEED).fit_transform(emb_raw)

    # UMAP on corrected latent
    emb_corr = np.asarray(adata.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
    umap_corr = UMAP(n_components=2, random_state=SEED).fit_transform(emb_corr)

    batch = adata.obs[BATCH_COL].astype(str)
    diag  = adata.obs[DIAGNOSIS_COL].astype(str)

    sns.set_theme(style="whitegrid", font_scale=1.0)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        f"{test_cohort} — Before vs After cVAE Correction  ({adata.n_obs} samples)",
        fontsize=14, fontweight="bold", y=1.02,
    )

    batch_pal = sns.color_palette("husl", n_colors=batch.nunique())
    diag_pal  = sns.color_palette("Set2", n_colors=diag.nunique())

    plot_umap_panel(umap_raw,  batch, "Before — RNA Batch",  axes[0, 0], batch_pal)
    plot_umap_panel(umap_corr, batch, "After — RNA Batch",   axes[0, 1], batch_pal)
    plot_umap_panel(umap_raw,  diag,  "Before — Diagnosis",  axes[1, 0], diag_pal)
    plot_umap_panel(umap_corr, diag,  "After — Diagnosis",   axes[1, 1], diag_pal)

    for ax in axes.flat:
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")

    plt.tight_layout()
    plt.show()

    del adata
else:
    logger.warning("No processed cohorts found for visualization.")

## 5. Quantitative Check — Silhouette Scores

In [ ]:
from sklearn.metrics import silhouette_score

rows = []
for name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_DIR / f"{name}.adata.zarr"
    if not p.exists():
        continue

    adata = load_cohort(p)

    emb_raw  = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    emb_corr = np.asarray(adata.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
    batch = adata.obs[BATCH_COL].astype(str).values
    diag  = adata.obs[DIAGNOSIS_COL].astype(str).values

    # Silhouette on batch (lower = better mixing = correction works)
    sil_batch_raw  = silhouette_score(emb_raw,  batch) if len(set(batch)) > 1 else np.nan
    sil_batch_corr = silhouette_score(emb_corr, batch) if len(set(batch)) > 1 else np.nan

    # Silhouette on diagnosis (higher = biology preserved)
    sil_diag_raw  = silhouette_score(emb_raw,  diag) if len(set(diag)) > 1 else np.nan
    sil_diag_corr = silhouette_score(emb_corr, diag) if len(set(diag)) > 1 else np.nan

    rows.append({
        "Cohort": name,
        "Samples": adata.n_obs,
        "Sil_Batch_Before": round(sil_batch_raw, 4),
        "Sil_Batch_After": round(sil_batch_corr, 4),
        "Sil_Batch_Δ": round(sil_batch_corr - sil_batch_raw, 4),
        "Sil_Diag_Before": round(sil_diag_raw, 4),
        "Sil_Diag_After": round(sil_diag_corr, 4),
        "Sil_Diag_Δ": round(sil_diag_corr - sil_diag_raw, 4),
    })

    del adata

if rows:
    df_sil = pd.DataFrame(rows)
    display(df_sil)
    logger.info("\nInterpretation:")
    logger.info("  Sil_Batch_Δ < 0 → better batch mixing (good)")
    logger.info("  Sil_Diag_Δ ~ 0 or > 0 → biology preserved (good)")
else:
    logger.warning("No processed cohorts found.")